# Testing Kwok Orchestration with Ko2Cube

This notebook demonstrates how to spin up a Kwok cluster, add nodes (Spot and On-Demand), deploy applications using native Deployments smoothly managed by the Kuberenetes kube-controller-manager, and simulate failures and spot interruptions.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('..'))

from server.kwok.cluster import Cluster

# 1. Create a simulated cluster
cluster = Cluster(name="demo-cluster", region="test")
cluster.create()

Creating cluster 'demo-cluster' (region=test, port=34099)...
Executing: kwokctl create cluster --name demo-cluster --kube-apiserver-port 34099


{"time":"2026-04-05T04:45:57.640141+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":304},"msg":"Cluster is creating","cluster":"demo-cluster"}
{"time":"2026-04-05T04:45:58.155354+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":311},"msg":"Cluster is created","cluster":"demo-cluster","elapsed":{"nanosecond":515634875,"human":"515.634875ms"}}
{"time":"2026-04-05T04:45:58.162426+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":344},"msg":"Cluster is starting","cluster":"demo-cluster"}


Cluster 'demo-cluster' created successfully!


{"time":"2026-04-05T04:45:58.754467+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/create/cluster.runE","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/create/cluster/cluster.go","line":349},"msg":"Cluster is started","cluster":"demo-cluster","elapsed":{"nanosecond":592030625,"human":"592.030625ms"}}


In [3]:
from server.kwok.config import load_kwok_kubeconfig
load_kwok_kubeconfig("demo-cluster")

Loaded kubeconfig for cluster 'demo-cluster' from dict.


{'apiVersion': 'v1',
 'clusters': [{'cluster': {'certificate-authority-data': 'LS0tLS1CRUdJTiBDRVJUSUZJQ0FURS0tLS0tCk1JSUM0ekNDQWN1Z0F3SUJBZ0lCQURBTkJna3Foa2lHOXcwQkFRc0ZBREFTTVJBd0RnWURWUVFERXdkcmQyOXIKTFdOaE1DQVhEVEkyTURRd016SXpNVFUxTjFvWUR6SXhNall3TXpFeE1qTXhOVFUzV2pBU01SQXdEZ1lEVlFRRApFd2RyZDI5ckxXTmhNSUlCSWpBTkJna3Foa2lHOXcwQkFRRUZBQU9DQVE4QU1JSUJDZ0tDQVFFQXFGRUhRcEgvClBIK2hObzRtVWxDcUNQQlUrT0QwZjhWOW44SjRxL1d3WU15M1RlSjdzOTNYbFZEWkRGYmpRcUxGVUVKYUxhbmQKUlZ3YmJoNUVWeVVJRjlCaGhvNnlTcXd5QlRHNmRreHd6WGVXSWp0OVJNeTNtRTZkVjFUY2daanYzb0ZRamZzNQp2NzVTL2hGdllYNTFFTWNMYzZLaVR4anhFbEcxTk9oYTlkTmlaVGkwbUFZUStyeWxjaFpyMHc5Vld0VmloN3JLCnh0Vng5Wm1vYXBrOC9UVWp4S0NlS0tuQ0pqanMrTkxzZmxBRmZ3THN3Vmk3OVkxY0JsMVdpMTZlbDlacVI4THYKN2J4aHRKL0JNMUhSaGQrcER0SnpidWYzeEM5Y2NreFQyWUlWOHdMNEVrQktXSU9pZHFRK2pMWnNOSmVzd2FteApMcklBRlBQdWZIWGMrd0lEQVFBQm8wSXdRREFPQmdOVkhROEJBZjhFQkFNQ0FxUXdEd1lEVlIwVEFRSC9CQVV3CkF3RUIvekFkQmdOVkhRNEVGZ1FVSWR2a0V1Yi83d0gyNVR2OGdsbEZFMnVCemlzd0RRWUpLb1pJaHZjTkFRRUwKQlFBRGdnRUJBQklGN

In [2]:
# 2. Spawn a Mix of On-Demand and Spot Nodes
node_regular = cluster.add_node(name="node-main", instance_type="m5.large", capacity_type="ON_DEMAND")
node_spot = cluster.add_node(name="node-spot-1", instance_type="c5.xlarge", capacity_type="SPOT")

print("Tracked python node references in Cluster:", list(cluster.kwok_nodes.keys()))

# Verify from Kubernetes API
for n in cluster.get_nodes():
    print(f"API Node Name: {n['name']}, Region: {n['region']}, Instance: {n['instance_type']}")

Loaded kubeconfig for cluster 'demo-cluster' from dict.
Creating node 'node-main' (m5.large, test)...
Node 'node-main' created successfully!
Loaded kubeconfig for cluster 'demo-cluster' from dict.
Creating node 'node-spot-1' (c5.xlarge, test)...
Node 'node-spot-1' created successfully!
Tracked python node references in Cluster: ['node-main', 'node-spot-1']
Loaded kubeconfig for cluster 'demo-cluster' from dict.
API Node Name: node-main, Region: test, Instance: m5.large
API Node Name: node-spot-1, Region: test, Instance: c5.xlarge


In [4]:
# 3. Deploy an App using Native Deployments
deployment = cluster.add_deployment(
    name="ai-agent-service", 
    replicas=3, 
    image="agent-image:v1"
)

import time
print("Waiting 3 seconds for kube-controller-manager to spin up the ReplicaSet...")
time.sleep(3) 

print("\nTracked Deployments:", list(cluster.kwok_deployments.keys()))
pods = cluster.get_pods()
print(f"\nTotal Pods Running in API: {len(pods)}")
for p in pods:
    print(f"- {p['name']} on {p['node_name']} (Phase: {p['phase']})")

Loaded kubeconfig for cluster 'demo-cluster' from dict.
Creating deployment 'ai-agent-service' with 3 replicas...
Deployment 'ai-agent-service' created successfully!
Waiting 3 seconds for kube-controller-manager to spin up the ReplicaSet...

Tracked Deployments: ['ai-agent-service']
Loaded kubeconfig for cluster 'demo-cluster' from dict.

Total Pods Running in API: 7
- cyclops-ctrl-b9d6cbc4-fjnf8 on node-main (Phase: Running)
- cyclops-ui-6f767f678c-8d7h4 on node-spot-1 (Phase: Running)
- ai-agent-service-78b5d5db8b-kc7w9 on node-spot-1 (Phase: Running)
- ai-agent-service-78b5d5db8b-pmsw7 on node-spot-1 (Phase: Running)
- ai-agent-service-78b5d5db8b-tnllp on node-main (Phase: Running)
- dashboard-metrics-scraper-5bd45c9dd6-98t64 on node-main (Phase: Running)
- kubernetes-dashboard-79cbcf9fb6-j2j4s on node-spot-1 (Phase: Running)


In [6]:
# 4. Simulate a Rollout update
print("Patching deployment image to v2...")
deployment.apply_new_rollout("agent-image:v2")
print("\nWait a moment, then check cluster.get_pods() below to see the pods replaced natively!")

Patching deployment image to v2...
Loaded kubeconfig for cluster 'demo-cluster' from dict.
Patching deployment 'ai-agent-service' with image 'agent-image:v2'...
Rollout triggered for 'ai-agent-service'.

Wait a moment, then check cluster.get_pods() below to see the pods replaced natively!


In [7]:
time.sleep(5)
for p in cluster.get_pods():
    print(f"- {p['name']} on {p['node_name']} (Phase: {p['phase']})")

Loaded kubeconfig for cluster 'demo-cluster' from dict.
- cyclops-ctrl-b9d6cbc4-fjnf8 on node-main (Phase: Running)
- cyclops-ui-6f767f678c-8d7h4 on node-spot-1 (Phase: Running)
- ai-agent-service-cc9b77c-4c55b on node-main (Phase: Running)
- ai-agent-service-cc9b77c-4qbqz on node-spot-1 (Phase: Running)
- ai-agent-service-cc9b77c-knfqx on node-spot-1 (Phase: Running)
- dashboard-metrics-scraper-5bd45c9dd6-98t64 on node-main (Phase: Running)
- kubernetes-dashboard-79cbcf9fb6-j2j4s on node-spot-1 (Phase: Running)


In [8]:
# 5. Simulate Spot Interruption
def my_callback():
    print("\n*** AGENT CALLBACK FIRED: Spot Node Interrupted and Deleted! ***\n")

node_spot.simulate_spot_interruption(delay_seconds=2, callback=my_callback)



[Spot Interruption] Terminating node 'node-spot-1'...
Loaded kubeconfig for cluster 'demo-cluster' from dict.
Deleting node 'node-spot-1'...
Node 'node-spot-1' deleted.

*** AGENT CALLBACK FIRED: Spot Node Interrupted and Deleted! ***



In [ ]:
# Verify that the Python Cluster state properly evicted the Spot node via the deletion callback
time.sleep(3)
print("\nRemaining Tracked Nodes in Python:", list(cluster.kwok_nodes.keys()))

In [ ]:
# Create a parallelized batch job
job = cluster.add_job(
    name="ai-training-job",
    node_name="node-main",               # The node you want it to run on
    active_deadline_seconds=300,         # Max age of the job before it fails
    completions=5,                       # We need 5 total pod completions
    parallelism=2,                       # Run 2 pods concurrently
    cpu="500m",
    duration="30s",   # Pod stays Running for 30 s                           
    memory="512Mi",
    backoff_limit=2                      # Fail the job entirely if 2 pods crash
)

import time
time.sleep(2) # Give kube-controller a second to spin them up

# Verify via Python tracking:
print("Tracked Jobs:", list(cluster.kwok_jobs.keys()))

# Verify via your Python Kubernetes wrapper:
for j in cluster.get_jobs():
    print(f"Job Name: {j['name']} | Status: {j['succeeded']}/{j['completions']} completed")


Loaded kubeconfig for cluster 'demo-cluster' from dict.
Creating job 'ai-training-job' on node 'node-main'...
Job 'ai-training-job' created successfully!
Tracked Jobs: ['ai-training-job']
Loaded kubeconfig for cluster 'demo-cluster' from dict.
Job Name: ai-training-job | Status: 4/5 completed


In [10]:
# 6. Cleanup
cluster.delete()

Deleting cluster 'demo-cluster'...


{"time":"2026-04-05T05:38:02.148239+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":121},"msg":"Cluster is stopping","cluster":"demo-cluster"}
{"time":"2026-04-05T05:38:02.543599+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":126},"msg":"Cluster is stopped","cluster":"demo-cluster","elapsed":{"nanosecond":395774125,"human":"395.774125ms"}}
{"time":"2026-04-05T05:38:02.543962+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":132},"msg":"Cluster is deleting","cluster":"demo-cluster"}


Cluster 'demo-cluster' deleted.


{"time":"2026-04-05T05:38:02.932896+05:30","level":"INFO","source":{"function":"sigs.k8s.io/kwok/pkg/kwokctl/cmd/delete/cluster.deleteCluster","file":"/private/tmp/kwok-20251226-5041-tfsm21/kwok-0.7.0/pkg/kwokctl/cmd/delete/cluster/cluster.go","line":148},"msg":"Cluster is deleted","cluster":"demo-cluster","elapsed":{"nanosecond":388927500,"human":"388.9275ms"}}
